# HW1. Numerical Programming with NumPy

## Overview

In this assignment you will implement a complete **K-Nearest Neighbors (KNN)** classification pipeline using only NumPy. Each function targets a specific NumPy skill from Lab 1-2: broadcasting for pairwise distance computation, `argsort` for neighbor retrieval, fancy indexing for feature gathering, and reductions for majority-vote prediction.

You will implement four functions inside `hw1_knn.py`. A separate file `test_knn.py` contains doctests you can run locally to check your work before submitting.

## Background: K-Nearest Neighbors

KNN is one of the simplest supervised learning algorithms. Given a labeled dataset of $N$ points and a new unlabeled query point, KNN finds the $K$ closest points in the dataset (measured by Euclidean distance) and predicts the query's label by majority vote among those $K$ neighbors.

The **Euclidean distance** between two $F$-dimensional vectors $\mathbf{a}$ and $\mathbf{b}$ is

$$
d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{f=1}^{F} (a_f - b_f)^2}
$$

Given $N$ data points in matrix $\mathbf{X} \in \mathbb{R}^{N \times F}$ and $Q$ query points in matrix $\mathbf{Z} \in \mathbb{R}^{Q \times F}$, you need to compute the full $Q \times N$ distance matrix where entry $(q, n)$ stores $d(\mathbf{z}_q, \mathbf{x}_n)$.

The KNN pipeline proceeds in three stages:

1. Compute all pairwise distances between queries and data points.
2. For each query, identify the indices of the $K$ nearest data points.
3. Gather the neighbors' labels and predict via majority vote.

> #### 📝 NumPy Skills Exercised
> Each function in this assignment maps directly to a core NumPy concept from Lab 1-2. `euclidean_distances` requires **broadcasting** and **reductions** (Sections 6, 9). `find_k_nearest_indices` uses **argsort** and **slicing** (Sections 2, 4). `calc_k_nearest_neighbors` combines the above with **fancy indexing** (Section 2). `predict_knn_classification` uses fancy indexing plus `np.bincount` / `np.argmax` for counting-based reductions.


## Problems

### Problem Definition

Complete the four `TODO` functions in `hw1_knn.py`. Each function builds on the previous one, so implement them in order. `test_knn.py` provides doctests for each function. Do not modify it.

Throughout this assignment, the doctests use a simple 2D dataset of $N = 4$ points lying on the unit circle, with $Q = 2$ query points slightly inside it:

$$
\mathbf{X} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ -1 & 0 \\ 0 & -1 \end{bmatrix}, \quad
\mathbf{Z} = \begin{bmatrix} 0.9 & 0 \\ 0 & -0.9 \end{bmatrix}
$$

The **Finding Nearest Neighbors via Euclidean Distance** pipeline flows as follows for $K = 3$:

1. **`euclidean_distances`** produces a $(2, 4)$ distance matrix. For example, the distance from query $\mathbf{z}_0 = [0.9,\; 0]$ to data point $\mathbf{x}_0 = [1,\; 0]$ is $0.1$.
2. **`find_k_nearest_indices`** sorts each row and returns the first $K$ column indices: query 0 gets neighbors at indices $[0, 1, 3]$.
3. **`calc_k_nearest_neighbors`** gathers the actual feature vectors at those indices, producing a $(2, 3, 2)$ array.
4. **`predict_knn_classification`** looks up each neighbor's label, counts votes, and returns the winning class per query.

Keep this example in mind as you implement each function below.

### Function 1: `euclidean_distances`

Compute the full pairwise Euclidean distance matrix between data and query points **without any Python loops**. The key idea is to use `np.newaxis` so that NumPy's broadcasting computes all $Q \times N$ differences in one shot.
````{.python filename="hw1_knn.py"}
def euclidean_distances(data_NF, query_QF):
    # Input:  data_NF  shape (N, F)
    #         query_QF shape (Q, F)
    # Output: dist_QN  shape (Q, N)
````

**Strategy.** Reshape `query_QF` to `(Q, 1, F)` and `data_NF` to `(1, N, F)`. Their difference broadcasts to shape `(Q, N, F)`. Square each element, sum over the last axis (features), and take the square root.

> #### 💡 Broadcasting Alignment
> When you write `query_QF[:, np.newaxis, :] - data_NF[np.newaxis, :, :]`, NumPy aligns the shapes from the right: `(Q, 1, F)` and `(1, N, F)` are compatible because every dimension pair is either equal or one of them is 1. The result has shape `(Q, N, F)`, giving you the per-feature difference for every query-data pair at once.


### Function 2: `find_k_nearest_indices`

Given a distance matrix of shape `(Q, N)`, return the indices of the $K$ closest data points for each query.
````{.python filename="hw1_knn.py"}
def find_k_nearest_indices(dist_QN, K):
    # Input:  dist_QN     shape (Q, N)
    #         K           int
    # Output: indices_QK  shape (Q, K), dtype int
````

**Strategy.** Apply `np.argsort` along `axis=1` to sort each row of distances. Then slice the first $K$ columns. NumPy's `argsort` is stable, so ties (equal distances) are broken by smaller index first.


### Function 3: `calc_k_nearest_neighbors`

Combine the previous two functions, then use fancy indexing to gather the actual feature vectors of the neighbors.
````{.python filename="hw1_knn.py"}
def calc_k_nearest_neighbors(data_NF, query_QF, K=1):
    # Input:  data_NF    shape (N, F)
    #         query_QF   shape (Q, F)
    #         K          int
    # Output: neighb_QKF shape (Q, K, F)
````

**Strategy.** Call `euclidean_distances`, then `find_k_nearest_indices`, then index into `data_NF` with the resulting integer array. When you write `data_NF[indices_QK]` where `indices_QK` has shape `(Q, K)`, NumPy replaces each index with the corresponding row of `data_NF`, producing shape `(Q, K, F)` directly.


### Function 4: `predict_knn_classification`

Given integer class labels for all data points and the neighbor indices for each query, predict each query's class by majority vote.
````{.python filename="hw1_knn.py"}
def predict_knn_classification(labels_N, nearest_indices_QK):
    # Input:  labels_N            shape (N,), dtype int
    #         nearest_indices_QK  shape (Q, K), dtype int
    # Output: predictions_Q       shape (Q,), dtype int
````

**Strategy.** Use fancy indexing `labels_N[nearest_indices_QK]` to get a `(Q, K)` array of neighbor labels. Then loop over queries: for each row, call `np.bincount` to count occurrences of each class and `np.argmax` to pick the most frequent one. Since `np.argmax` returns the first occurrence of the maximum value, ties are automatically broken in favor of the smallest class label.

> #### ❗ Rules You Must Follow
> Do not change function signatures or docstrings. Use only `numpy` (already imported). Do not add extra `import` statements. Make sure no `raise NotImplementedError` lines remain in your submitted code. The autograder compares return values exactly, so match the expected shapes and dtypes.


## Running Doctests

Place `hw1_knn.py` and [test_knn.py]{.kbd} in the same directory, then run from the terminal:
````{.bash filename="terminal"}
python -m doctest test_knn.py -v
````

If all tests pass you will see output ending with:
````{.text}
14 tests in 1 items.
14 passed and 0 failed.
Test passed.
````

You can also test from inside a notebook:
```{.python}
import doctest
import test_knn
doctest.testmod(test_knn, verbose=True)
```

## Submission

Upload only `hw1_knn.py` to eCampus. Do not rename the file. Do not submit the notebook or `test_knn.py`. The autograder will run both the public doctests shown above and additional hidden test cases covering edge cases such as K=N, single-query inputs, higher-dimensional features, and multi-class labels.

> #### ⚠️ Common Mistakes
> Forgetting to remove `raise NotImplementedError` is the most frequent cause of zero scores. Also watch out for returning the wrong shape (e.g., `(Q, N)` instead of `(Q, K)` in `find_k_nearest_indices`) and using Python loops where broadcasting is expected (the hidden tests may include timing checks for `euclidean_distances`).